# LO benchmark results

Load multi-model results for the linear-optimization (LO / LP) benchmark into one dataframe, then show a **score table**: rows = questions, columns = LLMs.

Each cell is `1` if the model's JSON `cost` was within 1% of the keyed objective, else `0`.

**Download in this notebook:** the load cell sets `DOWNLOAD_KAGGLE_RUNS = True` and pulls runs via the Kaggle CLI into `data/kaggle_runs/lo-normative-accuracy-5`. Requires `kaggle auth login` once.

If download is off and files are missing, the cell will still try to download automatically.

Or from a local sandbox / merged CSV: set `LOAD_FROM_KAGGLE = False` and put `*.run.json` or `*merged*.csv` under `data/lp/` or one of the sandbox candidate folders.

In [ ]:
# ! \src\projects\sceptical_llms\.venv\Scripts\python.exe -m kaggle auth logina/

The system cannot find the path specified.


In [3]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "lp").is_dir():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import benchmarks.kaggle_runs as kaggle_runs
import benchmarks.lp_rate as lp_rate

importlib.reload(lp_rate)
importlib.reload(kaggle_runs)

from benchmarks.kaggle_runs import (
    DEFAULT_LO_TASK_SLUG,
    download_task_runs,
    load_base_rate_run_rows_from_tree,
    merged_lo_results_from_kaggle_runs,
)
from benchmarks.lp_rate import write_merged_results_csv

# --- knobs ---
LOAD_FROM_KAGGLE = True
DOWNLOAD_KAGGLE_RUNS = True  # download via Kaggle CLI into data/kaggle_runs/...
# Task published by %choose lo_normative_accuracy_5
KAGGLE_TASK_SLUG = DEFAULT_LO_TASK_SLUG  # "lo-normative-accuracy-5"
assert KAGGLE_TASK_SLUG == "lo-normative-accuracy-5", KAGGLE_TASK_SLUG

# Sandbox / alternate run folders (first existing wins when LOAD_FROM_KAGGLE).
SANDBOX_CANDIDATES = [
    ROOT / "data" / "kaggle_runs" / "lo-normative-accuracy-5",
    Path("/kaggle/working") / "lp_benchmark",
    Path("/kaggle/working"),
]

BENCHMARK_CSV = ROOT / "data" / "lp" / "benchmark.csv"
MERGED_DIR = ROOT / "data" / "lp"


def _first_existing(paths: list[Path]) -> Path | None:
    for path in paths:
        if path.is_dir() and (
            any(path.rglob("*.run.json")) or any(path.glob("*merged*.csv"))
        ):
            return path
    return None


def _load_merged_from_run_tree(runs_dir: Path) -> tuple[pd.DataFrame, str]:
    try:
        merged_rows = merged_lo_results_from_kaggle_runs(
            runs_dir,
            benchmark_path=BENCHMARK_CSV,
            fill_missing=False,
        )
    except ValueError:
        # Fallback for sandbox trees that do not match the task-slug filter path.
        run_rows = load_base_rate_run_rows_from_tree(runs_dir)
        if not run_rows:
            raise
        out_csv = MERGED_DIR / "lo_merged_results.csv"
        write_merged_results_csv(
            run_rows,
            out_csv,
            benchmark_path=BENCHMARK_CSV,
        )
        merged_rows = pd.read_csv(out_csv).to_dict(orient="records")
    return pd.DataFrame(merged_rows), f"run tree ({runs_dir})"


if LOAD_FROM_KAGGLE:
    default_runs = ROOT / "data" / "kaggle_runs" / KAGGLE_TASK_SLUG
    if DOWNLOAD_KAGGLE_RUNS:
        print(f"Downloading {KAGGLE_TASK_SLUG} -> {default_runs}")
        default_runs.mkdir(parents=True, exist_ok=True)
        download_task_runs(KAGGLE_TASK_SLUG, default_runs)

    # Prefer the _2 download dir; do not fall back to lo-normative-accuracy (v1).
    runs_dir = _first_existing([default_runs, *SANDBOX_CANDIDATES])
    if runs_dir is None and not DOWNLOAD_KAGGLE_RUNS:
        print(f"No local runs found; downloading {KAGGLE_TASK_SLUG} -> {default_runs}")
        default_runs.mkdir(parents=True, exist_ok=True)
        download_task_runs(KAGGLE_TASK_SLUG, default_runs)
        runs_dir = _first_existing([default_runs, *SANDBOX_CANDIDATES])

    if runs_dir is None:
        raise FileNotFoundError(
            "No LO run results found after download attempt. Looked under:\n  - "
            + "\n  - ".join(str(p) for p in [default_runs, *SANDBOX_CANDIDATES])
            + f"\nManual download:\n  python -m kaggle benchmarks tasks download "
            f"{KAGGLE_TASK_SLUG} -o {default_runs}"
        )

    if any(runs_dir.rglob("*.run.json")):
        df, data_source = _load_merged_from_run_tree(runs_dir)
    else:
        merged_csv = sorted(
            runs_dir.glob("*merged*.csv"),
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )[0]
        df = pd.read_csv(merged_csv)
        data_source = str(merged_csv)
else:
    merged_candidates = sorted(
        list(MERGED_DIR.glob("*merged*.csv"))
        + list(MERGED_DIR.glob("lo_merged_results*.csv")),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    # De-dupe while preserving mtime order.
    seen: set[Path] = set()
    unique: list[Path] = []
    for path in merged_candidates:
        if path not in seen:
            seen.add(path)
            unique.append(path)
    if not unique:
        raise FileNotFoundError(
            f"Missing merged results under {MERGED_DIR}. "
            "Set LOAD_FROM_KAGGLE=True or run benchmark/lo-benchmark.ipynb."
        )
    merged_csv = unique[0]
    df = pd.read_csv(merged_csv)
    data_source = str(merged_csv)

if "score" not in df.columns:
    raise KeyError("Merged data must include 'score'.")

df["score_value"] = df["score"].astype(str).str.lower().eq("true").astype(int)
if "naive_lp_confusion" in df.columns:
    df["naive_value"] = (
        df["naive_lp_confusion"].astype(str).str.lower().eq("true").astype(int)
    )

print("Source:", data_source)
print("Task slug:", KAGGLE_TASK_SLUG)
print("Rows:", len(df))
print("Models:", sorted(df["model"].dropna().unique()))
print("Questions:", sorted(df["example_id"].unique()) if "example_id" in df else "?")
df.head()


Source: run tree (c:\src2\sceptical-llms\data\kaggle_runs\lo-normative-accuracy-5)
Task slug: lo-normative-accuracy-5
Rows: 150
Models: ['anthropic/claude-haiku-4-5@20251001', 'anthropic/claude-opus-4-8@default', 'anthropic/claude-opus-5@default', 'anthropic/claude-sonnet-4-6@default', 'google/gemini-2.5-flash', 'google/gemini-3-flash-preview', 'google/gemini-3.5-flash', 'google/gemini-3.6-flash', 'openai/gpt-5.6-sol', 'openai/gpt-5.6-terra']
Questions: ['carpenter_furniture__integrality__explicit__json', 'carpenter_furniture__integrality__json', 'charter_buses__integrality__explicit__json', 'charter_buses__integrality__json', 'fund_allocation__nonnegativity__explicit__json', 'fund_allocation__nonnegativity__json', 'gift_baskets__both__explicit__json', 'gift_baskets__both__json', 'pottery_studio__integrality__explicit__json', 'pottery_studio__integrality__json', 'print_shop__integrality__explicit__json', 'print_shop__integrality__json', 'warehouse_shipping__nonnegativity__explicit__jso

,example_id,vignette_name,failure_mode,condition,problem_type,intersection_size,response_type,has_statistics,variant,prompt,...,parsed_confidence,comment_line,scoring_type,parseable,score,naive_lp_confusion,parsed_objective,parsed_solution,score_value,naive_value
0,carpenter_furniture__integrality__explicit__json,carpenter furniture,integrality,explicit,explicit_constraints,,json,true,json,You are an operations consultant. Your task is...,...,,,open,true,true,false,200,"{""bookcases"": 0, ""desks"": 4}",1,0
1,carpenter_furniture__integrality__explicit__json,carpenter furniture,integrality,explicit,explicit_constraints,,json,true,json,You are an operations consultant. Your task is...,...,,,open,true,true,false,200,"{""bookcases"": 0, ""desks"": 4}",1,0
2,carpenter_furniture__integrality__explicit__json,carpenter furniture,integrality,explicit,explicit_constraints,,json,true,json,You are an operations consultant. Your task is...,...,,,open,true,true,false,200,"{""bookcases"": 0, ""desks"": 4}",1,0
3,carpenter_furniture__integrality__explicit__json,carpenter furniture,integrality,explicit,explicit_constraints,,json,true,json,You are an operations consultant. Your task is...,...,,,open,true,true,false,200,"{""bookcases"": 0, ""desks"": 4}",1,0
4,carpenter_furniture__integrality__explicit__json,carpenter furniture,integrality,explicit,explicit_constraints,,json,true,json,You are an operations consultant. Your task is...,...,,,open,true,false,false,225,"{""bookcases"": 2, ""desks"": 3}",0,0


## Score table: vignette × version × LLM

Rows are LO vignettes with a version label: JSON **implicit** / **explicit**, plus audit variants `needs_tacit_constraint` and `detects_tacit_violation`. Columns are models. Cell = mean keyed score (`1` = correct, `0` = incorrect).

In [4]:
plot_df = df.copy()
if "variant" in plot_df.columns:
    plot_df["version"] = plot_df.apply(
        lambda r: (
            str(r.get("condition") or "")
            if str(r.get("variant") or "") == "json"
            else str(r.get("variant") or "")
        ),
        axis=1,
    )
    index_cols: list[str] | str = ["vignette_name", "version"]
    version_order = {
        "implicit": 0,
        "explicit": 1,
        "needs_tacit_constraint": 2,
        "detects_tacit_violation": 3,
        "control": 4,
    }
    plot_df = plot_df.assign(
        _version_ord=plot_df["version"].map(version_order).fillna(99)
    ).sort_values(["vignette_name", "_version_ord"])
elif "condition" in plot_df.columns and "vignette_name" in plot_df.columns:
    index_cols = ["vignette_name", "condition"]
    condition_order = {"implicit": 0, "explicit": 1, "control": 2}
    plot_df = plot_df.assign(
        _condition_ord=plot_df["condition"].map(condition_order).fillna(99)
    ).sort_values(["vignette_name", "_condition_ord"])
elif "vignette_name" in plot_df.columns:
    index_cols = "vignette_name"
else:
    index_cols = "example_id"

score_table = (
    plot_df.pivot_table(
        index=index_cols,
        columns="model",
        values="score_value",
        aggfunc="mean",
    )
    .sort_index(axis=1)
)

# Keep vignette/version row order; append an overall mean row.
if isinstance(index_cols, list):
    ordered = plot_df[list(index_cols)].drop_duplicates()
    score_table = score_table.reindex(
        pd.MultiIndex.from_frame(ordered, names=list(index_cols))
    )
else:
    score_table = score_table.sort_index()

# Column means first (before adding the row-mean column), then append mean row
# with .values so MultiIndex loc does not mis-align into NaNs.
col_means = score_table.mean(axis=0)
score_table["mean"] = score_table.mean(axis=1)
mean_row = col_means.copy()
mean_row["mean"] = float(col_means.mean())
if isinstance(score_table.index, pd.MultiIndex):
    score_table.loc[("mean", ""), :] = mean_row.to_numpy()
else:
    score_table.loc["mean", :] = mean_row.to_numpy()

display_table = score_table.round(3)
display_table


model                          anthropic/claude-haiku-4-5@20251001  \
vignette_name       condition                                        
carpenter furniture implicit                                 0.000   
                    explicit                                 1.000   
charter buses       implicit                                 1.000   
                    explicit                                 0.000   
fund allocation     implicit                                 0.000   
                    explicit                                 0.000   
gift baskets        implicit                                 1.000   
                    explicit                                 1.000   
pottery studio      implicit                                 0.000   
                    explicit                                 1.000   
print shop          implicit                                 1.000   
                    explicit                                 1.000   
warehouse shipping  implicit                                 0.000   
                    explicit                                 0.000   
workshop vehicles   control                                  0.000   
mean                                                         0.467   

model                          anthropic/claude-opus-4-8@default  \
vignette_name       condition                                      
carpenter furniture implicit                               0.000   
                    explicit                               1.000   
charter buses       implicit                               1.000   
                    explicit                               1.000   
fund allocation     implicit                               0.000   
                    explicit                               0.000   
gift baskets        implicit                               1.000   
                    explicit                               1.000   
pottery studio      implicit                               0.000   
                    explicit                               1.000   
print shop          implicit                               0.000   
                    explicit                               0.000   
warehouse shipping  implicit                               1.000   
                    explicit                               1.000   
workshop vehicles   control                                0.000   
mean                                                       0.533   

model                          anthropic/claude-opus-5@default  \
vignette_name       condition                                    
carpenter furniture implicit                             1.000   
                    explicit                             1.000   
charter buses       implicit                             0.000   
                    explicit                             1.000   
fund allocation     implicit                             1.000   
                    explicit                             1.000   
gift baskets        implicit                             1.000   
                    explicit                             1.000   
pottery studio      implicit                             1.000   
                    explicit                             0.000   
print shop          implicit                             1.000   
                    explicit                             0.000   
warehouse shipping  implicit                             1.000   
                    explicit                             1.000   
workshop vehicles   control                              0.000   
mean                                                     0.733   

model                          anthropic/claude-sonnet-4-6@default  \
vignette_name       condition                                        
carpenter furniture implicit                                   0.0   
                    explicit                                   1.0   
charter buses       implicit                 

## Implicit − explicit score gap (vignette × LLM)

For each trap vignette, **implicit score minus explicit score** per model. Positive values mean the model scored higher when constraints were omitted; negative values mean spelling constraints out helped. The control vignette is omitted (no explicit parallel).

In [5]:
## Implicit − explicit score gap (vignette × LLM)

For each trap vignette, **JSON implicit score minus JSON explicit score** per model. Positive values mean the model scored higher when constraints were omitted; negative values mean spelling constraints out helped. Audit variants are omitted here.


model,anthropic/claude-haiku-4-5@20251001,anthropic/claude-opus-4-8@default,anthropic/claude-opus-5@default,anthropic/claude-sonnet-4-6@default,google/gemini-2.5-flash,google/gemini-3-flash-preview,google/gemini-3.5-flash,google/gemini-3.6-flash,openai/gpt-5.6-sol,openai/gpt-5.6-terra,mean
vignette_name,,,,,,,,,,,
carpenter furniture,-1.000,-1.000,0.000,-1.000,0.000,0.000,0.0,0.0,0.0,0.0,-0.300
charter buses,1.000,0.000,-1.000,-1.000,0.000,0.000,0.0,0.0,0.0,0.0,-0.100
fund allocation,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.0,0.0,0.0,0.000
gift baskets,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.0,0.0,0.0,0.000
pottery studio,-1.000,-1.000,1.000,-1.000,0.000,1.000,0.0,0.0,0.0,0.0,-0.100
print shop,0.000,0.000,1.000,0.000,0.000,0.000,0.0,0.0,0.0,0.0,0.100
warehouse shipping,0.000,0.000,0.000,0.000,-1.000,0.000,0.0,0.0,0.0,0.0,-0.100
mean,-0.143,-0.286,0.143,-0.429,-0.143,0.143,0.0,0.0,0.0,0.0,-0.071


In [7]:
_json = score_table.drop(columns=["mean"], errors="ignore")
if not isinstance(_json.index, pd.MultiIndex):
    raise ValueError("Expected MultiIndex on score_table")

# Prefer version labels from the new score table; fall back to condition.
level_names = list(_json.index.names)
level = "version" if "version" in level_names else "condition"
_json = _json[_json.index.get_level_values("vignette_name") != "mean"]
_implicit = _json.xs("implicit", level=level)
_explicit = _json.xs("explicit", level=level)

if "version" in plot_df.columns:
    vignette_order = (
        plot_df.loc[plot_df["version"].isin(["implicit", "explicit"]), "vignette_name"]
        .drop_duplicates()
        .tolist()
    )
else:
    vignette_order = (
        plot_df.loc[plot_df["condition"].isin(["implicit", "explicit"]), "vignette_name"]
        .drop_duplicates()
        .tolist()
    )

implicit_explicit_diff = _implicit.sub(_explicit).reindex(vignette_order)
implicit_explicit_diff = implicit_explicit_diff.sort_index(axis=1)
implicit_explicit_diff["mean"] = implicit_explicit_diff.mean(axis=1)

_col_means = implicit_explicit_diff.drop(columns=["mean"]).mean(axis=0)
_mean_row = _col_means.copy()
_mean_row["mean"] = float(_col_means.mean())
implicit_explicit_diff.loc["mean"] = _mean_row.to_numpy()

implicit_explicit_diff.round(3)


Wrote c:\src2\sceptical-llms\data\lp\lo_score_by_question_model.csv
Wrote c:\src2\sceptical-llms\data\lp\lo_implicit_minus_explicit_by_vignette_model.csv
